# z020 - Sacar ccajas_depositos (rota en 202105)

`ccajas_depositos` es NA en todo 202105 (estaba todo en cero). Eso también afecta sus lags y deltas
(por ejemplo, en 202106 el lag1 y la delta1 son NA). ¿Conviene sacarla del todo?

Versiones: **Con ccajas_depositos** (actual) vs **Sin ccajas_depositos** (se saca la variable, sus lags y deltas).
Cortes con 2 meses de distancia (como la competencia): A 202103 → 202105 y C 202104 → 202106 (control).
Además D: 202103 + 202104 → 202105 (ojo: el target de abril usa información de junio, futuro de mayo;
sirve para comparar con/sin, no como ganancia absoluta).
Modelo común: solo BAJA+2, trial 34 de z005 (538 árboles), 5 semillas.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Entrenar y evaluar ---
PREFIJO = "z020 sin ccajas_depositos"
esquemas = {"A: 202103 -> 202105": ([202103], 202105),
            "C: 202104 -> 202106": ([202104], 202106),
            "D: 202103+202104 -> 202105": ([202103, 202104], 202105)}
sin_ccajas = [v for v in variables if not (v == "ccajas_depositos" or v.startswith("ccajas_depositos_"))]
versiones = {"Con ccajas_depositos": variables, "Sin ccajas_depositos": sin_ccajas}
print("Variables con:", len(variables), "| sin:", len(sin_ccajas))

filas = []
for esquema, (meses, mes_valid) in esquemas.items():
    train = df[df["foto_mes"].isin(meses)]
    valid = df[df["foto_mes"] == mes_valid]
    g_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    for version, vars_v in versiones.items():
        datos = lgb.Dataset(train[vars_v], label=train["clase_binaria"], free_raw_data=False)
        for s in SEMILLAS:
            prob = lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).predict(valid[vars_v])
            acum = np.cumsum(g_cliente[np.argsort(prob)[::-1]])
            n = int(np.argmax(acum)) + 1
            filas.append({"esquema": esquema, "version": version, "semilla": s,
                          "ganancia_max_M": round(acum[n - 1] / 1e6, 1), "mejor_n": n,
                          "top12000_M": round(acum[11_999] / 1e6, 1)})
        print(esquema, "|", version, "listo", flush=True)
res = pd.DataFrame(filas)
tabla = res.groupby(["esquema", "version"], sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

Variables con: 761 | sin: 756


A: 202103 -> 202105 | Con ccajas_depositos listo


A: 202103 -> 202105 | Sin ccajas_depositos listo


C: 202104 -> 202106 | Con ccajas_depositos listo


C: 202104 -> 202106 | Sin ccajas_depositos listo


D: 202103+202104 -> 202105 | Con ccajas_depositos listo


D: 202103+202104 -> 202105 | Sin ccajas_depositos listo


In [4]:
# --- 4. Comparación ---
for esquema in esquemas:
    g = res[res["esquema"] == esquema].pivot(index="semilla", columns="version", values="ganancia_max_M")
    dif = g["Sin ccajas_depositos"] - g["Con ccajas_depositos"]
    print(f"{esquema}: sin - con = {dif.mean():+.1f} M, gana en {(dif > 0).sum()} de {len(dif)} semillas")
tabla

A: 202103 -> 202105: sin - con = +0.1 M, gana en 2 de 5 semillas
C: 202104 -> 202106: sin - con = -0.7 M, gana en 3 de 5 semillas
D: 202103+202104 -> 202105: sin - con = -2.6 M, gana en 0 de 5 semillas


ganancia_max_M       \
                                                          mean  std   
esquema                    version                                    
A: 202103 -> 202105        Con ccajas_depositos          252.6  1.7   
                           Sin ccajas_depositos          252.7  1.3   
C: 202104 -> 202106        Con ccajas_depositos          399.6  2.3   
                           Sin ccajas_depositos          398.9  2.2   
D: 202103+202104 -> 202105 Con ccajas_depositos          272.9  1.6   
                           Sin ccajas_depositos          270.3  2.5   

                                                top12000_M       mejor_n  \
                                                      mean  std     mean   
esquema                    version                                         
A: 202103 -> 202105        Con ccajas_depositos      229.7  2.8   7861.4   
                           Sin ccajas_depositos      233.6  4.2   7707.0   
C: 202104 -> 202106        Con ccajas_depositos      392.9  3.7  13212.0   
                           Sin ccajas_depositos      390.3  5.5  13429.8   
D: 202103+202104 -> 202105 Con ccajas_depositos      246.8  2.0   8677.4   
                           Sin ccajas_depositos      245.7  3.7   8755.8   

                                                         
                                                    std  
esquema                    version                       
A: 202103 -> 202105        Con ccajas_depositos  1122.2  
                           Sin ccajas_depositos   963.1  
C: 202104 -> 202106        Con ccajas_depositos  1072.1  
                           Sin ccajas_depositos  1078.6  
D: 202103+202104 -> 202105 Con ccajas_depositos   293.0  
                           Sin ccajas_depositos   372.7

In [5]:
# --- 5. Anotar ---
ruta_res = "resultados_experimentos.csv"
r = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
r = r[~r["descripcion"].str.startswith(PREFIJO)]
for (esquema, version), x in res.groupby(["esquema", "version"], sort=False):
    meses, mes_valid = esquemas[esquema]
    r = pd.concat([r, pd.DataFrame([{
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"{PREFIJO} esquema {esquema} | {version} (prom. 5 semillas)",
        "meses_train": str(meses), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params),
        "mejor_n": int(x["mejor_n"].mean()), "ganancia_max_M": round(x["ganancia_max_M"].mean(), 1),
        "ganancia_top12000_M": round(x["top12000_M"].mean(), 1)}])], ignore_index=True)
r["n_enviados"] = r["n_enviados"].astype("Int64")
r.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Anotado")

Anotado


## Conclusiones (08/10)
| Corte | Con | Sin | Sin − con |
|---|---|---|---|
| A: 202103 → 202105 | 252,6 | 252,7 | +0,1 M (2/5 semillas) |
| C: 202104 → 202106 | 399,6 | 398,9 | −0,7 M (3/5) |
| D: 202103+202104 → 202105 | 272,9 | 270,3 | −2,6 M (0/5) |
- Sacar `ccajas_depositos` **no cambia nada**: LightGBM maneja bien el NA de mayo.
- La dificultad de mayo (27-29% del ideal vs ~34% en junio, z018) **no se explica por esta variable**.
- Era para sacarnos la duda: la variable se deja como está.